# EDA: 정책 변수 검증
## Layer 1 — 정책 변수의 정의와 기준치를 데이터로 확정

## 목적

`01_data_mart.ipynb`에서 만든 마스터 테이블에 포함된 DTI, CIR, LTV가 정책 정의식과 일치하는지 검증하고, 각 변수 구간별 부도율과 승인, 거절 분포를 비교해 한도 산출 및 심사 기준에 사용할 수 있는지 확인하는 단계입니다. 단순 상관 확인이 아니라, 실제 정책 규칙에 넣을 수 있는 변수인지 검증하는 목적의 분석입니다.

이 노트북은 세 가지를 확인합니다.
1. DTI·CIR·LTV의 계산식이 실제 데이터 단위와의 적합성
2. 각 변수가 부도율과 갖고있는 방향성
3. 정책 규칙(`05_policy_simulation.ipynb`)에 쓸 임계값 설정 

이를 통해, 실제 한도 및 심사 규칙으로 전환 가능한지 검증합니다.

## Step 1. 대출 규모 분포 및 CIR(소득 대비 대출 배수) 확인

CIR(Credit-to-Income Ratio)은 "소득 대비 대출 배수"를 나타내는 지표입니다. DTI가 매달의 상환 부담(흐름, flow)을 보는 지표라면, CIR은 대출 원금 자체의 규모(재고, stock)를 보는 지표라는 점에서 역할이 다릅니다. 이는 상환 능력과 별개로 대출 규모 자체가 부도율과 연결되는지 확인하기 위한 1차 검증을 합니다.

Home Credit처럼 소득 대비 대출 배수를 통제하는 것이 중요한 unbanked 소비자 금융사에서는, CIR은 소득 대비 대출 원금의 크기를 직접 반영하므로 unbanked 고객군에서 한도 상한을 정하는 핵심 변수입니다. PD가 승인 여부를 가르는 지표라면, CIR은 승인 이후 어느 수준까지 노출을 허용할지 결정하는 지표로서 상한 설정의 근거가 됩니다.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

train = pd.read_csv('../data/train_raw.csv')

print("AMT_CREDIT 분포:")
print(train['AMT_CREDIT'].describe())

print("\nCREDIT_INCOME_RATIO(CIR) 분포:")
print(train['CIR'].describe())
print(train['CIR'].quantile([0.1, 0.25, 0.5, 0.75, 0.9]))

print("\n대출 종류별 AMT_CREDIT 중앙값:")
print(train.groupby('NAME_CONTRACT_TYPE')['AMT_CREDIT'].median())

AMT_CREDIT 분포:
count    2.460080e+05
mean     5.993915e+05
std      4.029081e+05
min      4.500000e+04
25%      2.700000e+05
50%      5.135310e+05
75%      8.086500e+05
max      4.050000e+06
Name: AMT_CREDIT, dtype: float64

CREDIT_INCOME_RATIO(CIR) 분포:
count    246008.000000
mean          3.959113
std           2.690285
min           0.004808
25%           2.018667
50%           3.267273
75%           5.166667
max          84.736842
Name: CIR, dtype: float64
0.10    1.332450
0.25    2.018667
0.50    3.267273
0.75    5.166667
0.90    7.487500
Name: CIR, dtype: float64

대출 종류별 AMT_CREDIT 중앙값:
NAME_CONTRACT_TYPE
Cash loans         542133.0
Revolving loans    270000.0
Name: AMT_CREDIT, dtype: float64


AMT_CREDIT 범위가 45,000~4,050,000으로 90배에 달하고, `Revolving loans`(신용카드형 반복 대출)의 중앙값이 `Cash loans`의 정확히 절반으로 상품 유형별 중앙값도 크게 갈립니다. 신청 금액의 분산이 크고 상품별 기준점 자체가 다르다는 것은, 단일 소득배수 공식 하나로 전체 상품을 일괄 커버하기 어렵다는 뜻입니다.

리볼빙은 반복 사용이 가능한 대출 구조라 리스크 관리 차원에서 한도를 낮게 운영하는 것이 일반적인데, 이 패턴이 데이터에서도 그대로 관찰됩니다. 즉 Home Credit은 대출 종류에 따라 한도 운영 철학 자체를 다르게 가져가고 있으며, 같은 CIR이라도 상품 구조에 따라 허용 한도가 다르게 설정돼야 한다는 방향을 시사합니다.

## Step 2. DTI 계산 검증

DTI(Debt-to-Income Ratio)는 `월 상환액 / 월 소득`으로 정의되는, "이 고객이 매달 벌어들이는 소득 중 얼마를 대출 상환에 쓰는가"를 나타내는 지표입니다. Step 1의 CIR이 대출 원금의 규모(재고)를 본다면, DTI는 매달 반복되는 상환 부담(흐름)을 봅니다. CIR로 대출 한도의 상한을 정해도 월 상환액이 소득 대비 과도하면 상환 불이행 위험이 커집니다. 그래서 DTI는 한도 산출과 별개로 실제 상환 부담을 점검하는 심사 단계의 보완 장치 역할을 합니다.

In [4]:
dti = train['AMT_ANNUITY'] / (train['AMT_INCOME_TOTAL'] / 12)
print("DTI 분포:")
print(dti.describe())
print(f"\n DTI 중앙값:", dti.median())

DTI 분포:
count    245996.000000
mean          2.171107
std           1.135732
min           0.002687
25%           1.376870
50%           1.954909
75%           2.748960
max          22.511579
dtype: float64

 DTI 중앙값: 1.954909090909091


DTI 중앙값이 1.95로 산출되었으나, 이를 곧바로 195%로 해석하는 것은 적절하지 않을 수 있어 계산식과 원본 컬럼의 단위를 재검증합니다.

In [6]:
sample = train[['AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_ANNUITY']].head(10).copy()
sample['DTI_월소득_12나눔'] = sample['AMT_ANNUITY'] / (sample['AMT_INCOME_TOTAL'] / 12)
sample['DTI_그대로나눔']    = sample['AMT_ANNUITY'] / sample['AMT_INCOME_TOTAL']
sample['상환개월수_추정']    = sample['AMT_CREDIT'] / sample['AMT_ANNUITY']
print(sample)

print(f"\n상환 기간(AMT_CREDIT/AMT_ANNUITY) 분포:")
repayment_months = train['AMT_CREDIT'] / train['AMT_ANNUITY']
print(repayment_months.describe())
print(repayment_months.quantile([0.75, 0.90, 0.95, 0.99]))

print(f"\nAMT_INCOME_TOTAL 분포:")
print(train['AMT_INCOME_TOTAL'].describe())
print(f"\nAMT_INCOME_TOTAL 중앙값 대비 AMT_CREDIT 중앙값 배수: "
      f"{train['AMT_CREDIT'].median() / train['AMT_INCOME_TOTAL'].median():.2f}배")

dti_raw_full = train['AMT_ANNUITY'] / train['AMT_INCOME_TOTAL']
print(f"\nDTI_그대로나눔(AMT_ANNUITY/AMT_INCOME_TOTAL) 전체 분포:")
print(dti_raw_full.describe())
print(f"1을 초과하는 비율(소득 전액 이상을 매달 상환한다는 뜻, "
      f"비정상이어야 정상): {(dti_raw_full > 1).mean():.2%}")

   AMT_CREDIT  AMT_INCOME_TOTAL  AMT_ANNUITY  DTI_월소득_12나눔  DTI_그대로나눔  \
0    269550.0           90000.0      16285.5      2.171400   0.180950   
1    269550.0           81000.0      11416.5      1.691333   0.140944   
2    291915.0           90000.0      25182.0      3.357600   0.279800   
3    855000.0          135000.0      45684.0      4.060800   0.338400   
4    135000.0           90000.0       6750.0      0.900000   0.075000   
5     79128.0          103500.0       5634.0      0.653217   0.054435   
6    361998.0          117000.0      19768.5      2.027538   0.168962   
7   1078200.0          135000.0      38200.5      3.395600   0.282967   
8   1078200.0          157500.0      34911.0      2.659886   0.221657   
9    783000.0          270000.0      22891.5      1.017400   0.084783   

    상환개월수_추정  
0  16.551534  
1  23.610564  
2  11.592209  
3  18.715524  
4  20.000000  
5  14.044728  
6  18.311860  
7  28.224761  
8  30.884249  
9  34.204836  

상환 기간(AMT_CREDIT/AMT_ANNUITY) 

`AMT_ANNUITY`와 `AMT_INCOME_TOTAL`의 단위를 원본 분포와 상환기간을 통해 재검증 했습니다. 상환기간 중앙값이 20개월로 현실적인 범위에 들어가 `AMT_ANNUITY`는 월 상환액으로 해석하는 것이 타당했으며, `AMT_ANNUITY` / `AMT_INCOME_TOTAL`의 중앙값이 0.163, 1 초과 비율이 0.01%로 나타나 `AMT_INCOME_TOTAL`도 월 소득 기준으로 정리되어 있음을 확인했습니다. 최초 계산식에서 12를 다시 적용한 부분은 단위 정합성에 맞지 않아 수정했습니다.

## Step 3. 수정된 DTI 분포 확인

In [7]:
print("DTI(수정) 분포:")
print(train['DTI'].describe())
print(train['DTI'].quantile([0.25, 0.5, 0.75, 0.9]))

DTI(수정) 분포:
count    245996.000000
mean          0.180926
std           0.094644
min           0.000224
25%           0.114739
50%           0.162909
75%           0.229080
max           1.875965
Name: DTI, dtype: float64
0.25    0.114739
0.50    0.162909
0.75    0.229080
0.90    0.301678
Name: DTI, dtype: float64


중앙값 0.163(16.3%), 75th percentile 0.229(22.9%)로 이제 현실적인 범위에 들어옵니다. 실무에서 DTI 상한을 통상 40%로 두는데, 이 데이터에서는 75th percentile조차 23%로 대부분 그 이하에 분포합니다.

`AMT_ANNUITY / AMT_CREDIT`의 중앙값이 0.05라는 것도 확인했습니다. 이는 대출 금액의 5%를 매달 상환한다는 뜻이며, 앞서 확인한 상환 기간 중앙값(20개월 ≈ 1/0.05)과 정확히 맞아떨어집니다. Home Credit의 대출이 짧은 만기의 소비자 금융 상품 구조임을 다시 확인해 줍니다.

## Step 4. LTV(대출/담보물가격 비율) 검증

LTV(Loan to Value)는 담보가치 대비 대출금 비중을 나타내는 지표로, 부도 시 담보 처분을 통한 회수 가능성을 기준으로 손실을 통제하는 핵심 변수입니다. 담보가치보다 과도하게 대출하면 회수 손실이 커질 수 있어, 실제 정책에서는 담보 유형과 상품 특성에 따라 LTV 상한을 관리합니다.

Home Credit처럼 특정 물품 구매 목적의 대출에서는 LTV를 구매 물품 가격 대비 대출금 비율로 해석할 수 있습니다. 이 값이 높을수록 실제 구매가 대비 대출 규모가 과도한지, 이자나 수수료 등 부대비용이 반영되어 있는지 점검하는 보조 리스크 지표로 활용할 수 있습니다.

In [8]:
print("LTV 분포:")
print(train['LTV'].describe())

ltv_over_1 = (train['LTV'] > 1).mean()
print(f"\nLTV > 1 비율: {ltv_over_1:.1%}")
print(f"LTV <= 1 비율: {1-ltv_over_1:.1%}")

print("\n대출 종류별 LTV 중앙값:")
print(train.groupby('NAME_CONTRACT_TYPE')['LTV'].median())

print(f"\nAMT_GOODS_PRICE 결측 비율: {train['AMT_GOODS_PRICE'].isna().mean():.1%}")

LTV 분포:
count    245788.000000
mean          1.123011
std           0.124180
min           0.150000
25%           1.000000
50%           1.118800
75%           1.198000
max           6.000000
Name: LTV, dtype: float64

LTV > 1 비율: 64.6%
LTV <= 1 비율: 35.4%

대출 종류별 LTV 중앙값:
NAME_CONTRACT_TYPE
Cash loans         1.132
Revolving loans    1.000
Name: LTV, dtype: float64

AMT_GOODS_PRICE 결측 비율: 0.1%


LTV 중앙값이 1.12이고 LTV가 1을 초과하는 비중이 64.6%로 높아, 일반적인 담보 기반 대출이라기보다 상품 구조 차이가 반영된 것으로 보입니다. 특히 Revolving loans에서 LTV가 정확히 1로 나타났는데, 이는 특정 물건 구매보다 한도 설정을 중심으로 운영되는 상품 특성상 `AMT_GOODS_PRICE` = `AMT_CREDIT`으로 기록되기 때문이라고 해석할 수 있습니다. 반면 `Cash loans`의 LTV 중앙값이 1.132로 1을 상회해, 일반적인 담보 기반 LTV 해석과는 다른 상품 구조가 반영되어 있음을 확인했습니다. 이는 이 데이터의 LTV를 담보비율로 읽기보다, **구매금액 대비 대출 규모와 부대비용 구조를 함께 반영하는 정책 변수**로 재해석해야 함을 시사합니다.

## Step 5. LTV와 실제 부도율의 관계

LTV를 비용 부담 지표로 재해석했다면, LTV가 높을수록 실제 상환 부담도 커져 부도율도 함께 올라가는지 확인합니다.

In [9]:
train['LTV_BIN'] = pd.cut(
    train['LTV'],
    bins=[0, 1.0, 1.1, 1.2, 1.5, float('inf')],
    labels=['≤1.0', '1.0~1.1', '1.1~1.2', '1.2~1.5', '>1.5']
)

print("LTV 구간별 부도율:")
print(train.groupby('LTV_BIN', observed=True)['TARGET'].mean())
print("\nLTV 구간별 건수:")
print(train.groupby('LTV_BIN', observed=True)['TARGET'].count())

LTV 구간별 부도율:
LTV_BIN
≤1.0       0.067722
1.0~1.1    0.049283
1.1~1.2    0.080553
1.2~1.5    0.115254
>1.5       0.136913
Name: TARGET, dtype: float64

LTV 구간별 건수:
LTV_BIN
≤1.0       86752
1.0~1.1    21772
1.1~1.2    85559
1.2~1.5    49638
>1.5        2067
Name: TARGET, dtype: int64


구간별 부도율은 `1.0~1.1` 구간에서 최저를 보였고, `1.2` 이상부터는 부도율이 8.0%에서 11.6%, 13.6%로 상승해 LTV 1.2를 기점으로 리스크가 커지는 패턴이 확인됐습니다. 특히 `1.0~1.1` 구간의 낮은 부도율은 LTV가 단순 담보비율이 아니라 상품 구조와 비용 부담을 함께 반영하는 변수일 가능성을 시사합니다. LTV가 `1.0`에 가까운 구간은 대출금과 물건 가격이 거의 일치해 상환 부담이 상대적으로 낮은 집단으로 해석되며, LTV `1.2`부터는 부도율이 뚜렷하게 상승해 정책상 변곡점으로 볼 수 있습니다. 즉, **1.2 이상 구간**은 부대비용이나 상품 구조상 부담이 리스크 증가와 함께 나타나는 **경험적 경계선**으로 해석할 수 있습니다.

## Step 6. DTI와 실제 부도율의 관계

LTV와 같은 방식으로, 수정된 DTI가 부도율을 설명하는지 확인합니다.

In [10]:
train['DTI_BIN'] = pd.cut(
    train['DTI'],
    bins=[0, 0.10, 0.20, 0.30, 0.40, float('inf')],
    labels=['<10%', '10~20%', '20~30%', '30~40%', '>40%']
)

print("DTI 구간별 부도율:")
print(train.groupby('DTI_BIN', observed=True)['TARGET'].mean())
print("\nDTI 구간별 건수:")
print(train.groupby('DTI_BIN', observed=True)['TARGET'].count())

DTI 구간별 부도율:
DTI_BIN
<10%      0.071843
10~20%    0.080454
20~30%    0.087436
30~40%    0.083062
>40%      0.080888
Name: TARGET, dtype: float64

DTI 구간별 건수:
DTI_BIN
<10%       45864
10~20%    116079
20~30%     59003
30~40%     18745
>40%        6305
Name: TARGET, dtype: int64


전체 부도율 약 8% 대비 DTI 구간별 부도율은 `7.2% ~ 8.8%` 범위로 큰 차이가 없었고, DTI가 높아질수록 부도율이 일관되게 증가하는 패턴은 관찰되지 않았습니다. 특히 `20 ~ 30%` 구간(8.8%)과 `30 ~ 40%` 구간(8.1%)의 차이도 단조 관계를 뒷받침하지 않았습니다.

이 결과는 심사 과정에서 이미 **고위험 고객이 일부 걸러졌거나**, 동일한 DTI라도 **절대 소득 수준 차이로 상환 여력이 달라졌을 가능성**을 시사합니다. 또한 **고객군 전반의 재정 취약성**이 유사해 DTI 단독 변별력이 제한적일 수도 있습니다.

그러므로 DTI는 독립적인 거절 기준으로는 약하지만, 고DTI 구간에서 상환 부담이 커질 가능성은 있어서 **PD와 함께 제한적으로 반영**합니다.

## Step 7. CIR(소득 대비 대출 배수) 실용 상한 도출

CIR은 "월 소득의 몇 배를 빌렸는가"를 나타냅니다. 이론적 상한을 DTI 상한과 상환 기간의 관계로 역산하고, 실제 데이터 분포와 비교해 실용적인 정책 상한을 정합니다.

In [11]:
print("CIR 분포:")
print(train['CIR'].describe())
print(train['CIR'].quantile([0.1, 0.25, 0.5, 0.75, 0.9]))

# 상환 기간 분포 (Step 2에서 계산한 것 재사용)
repayment_months = train['AMT_CREDIT'] / train['AMT_ANNUITY']
print("\n상환 기간 분포:")
print(repayment_months.quantile([0.75, 0.90, 0.95, 0.99]))

# 이론적 상한: DTI 상한(0.40) x 최대 상환기간
dti_cap = 0.40
max_months = repayment_months.quantile(0.99)
practical_months = repayment_months.quantile(0.90)

theoretical_cir_cap = dti_cap * max_months
practical_cir_cap   = 0.35 * practical_months

print(f"\n이론적 CIR 상한 (DTI 40% x 최대상환기간 {max_months:.0f}개월): {theoretical_cir_cap:.1f}")
print(f"실용적 CIR 상한 (DTI 35% x 90th 상환기간 {practical_months:.0f}개월): {practical_cir_cap:.1f}")

CIR 분포:
count    246008.000000
mean          3.959113
std           2.690285
min           0.004808
25%           2.018667
50%           3.267273
75%           5.166667
max          84.736842
Name: CIR, dtype: float64
0.10    1.332450
0.25    2.018667
0.50    3.267273
0.75    5.166667
0.90    7.487500
Name: CIR, dtype: float64

상환 기간 분포:
0.75    27.099985
0.90    34.063122
0.95    34.588021
0.99    37.793373
dtype: float64

이론적 CIR 상한 (DTI 40% x 최대상환기간 38개월): 15.1
실용적 CIR 상한 (DTI 35% x 90th 상환기간 34개월): 11.9


CIR 중앙값은 3.27배(월 소득의 3.27배를 대출), 90th percentile은 7.5배입니다.

상환기간과 DTI 상한을 결합해 역산하면 이론적 최대 한도는 월 소득의 약 18배이지만, 이는 극단적 상환기간을 가정한 값이라 실무 기준으로는 과도하게 느슨합니다. 따라서 90th percentile 상환기간 34개월과 보수적 DTI 35%를 적용해, 운영상 참고 가능한 CIR 실용 상한을 **약 12배**로 설정했습니다.

정책 구간은 데이터 분포를 반영해 저위험(≤3.0배, 중앙값 이하)/중위험(3.0~5.0배, 75th 근방)/고위험(5.0~7.5배, 90th 근방)/한도초과(7.5배 초과)로 나눕니다. 앞서 도출한 실용 상한(약 12배)은 이 구간 자체의 경계는 아니지만, 한도초과 구간 내에서도 절대 승인 불가로 볼 수 있는 상한선의 참고 기준으로 남겨둡니다. 실제 승인·추가심사·한도축소 같은 정책 판단으로의 연결은 `05_policy_simulation.ipynb`에서 다룹니다.

## 다음 단계에서는

이 노트북에서 검증한 세 변수는 각기 다른 역할로 확정됐습니다.

| 변수 | 계산식 | 부도율과의 관계 | 정책에서의 역할 |
|---|---|---|---|
| **LTV** | AMT_CREDIT / AMT_GOODS_PRICE | 1.2를 기점으로 뚜렷한 변곡점 | 독립적인 위험 신호로 사용 (기준: 1.2) |
| **DTI** | AMT_ANNUITY / AMT_INCOME_TOTAL | 제한적 반영 | PD 등 다른 지표와 결합했을 때만 보조 조건 |
| **CIR** | AMT_CREDIT / AMT_INCOME_TOTAL | 한도 규모의 절대적 상한 기준 | 한도 산출 시 상한 캡(실용 상한 약 12배) |

가장 중요한 발견은 정책 변수라고 해서 모두 동일한 방식으로 다룰 수는 없다는 점입니다. LTV처럼 단독으로도 비교적 뚜렷한 신호를 보이는 변수도 있지만, DTI처럼 겉보기에는 핵심 정책 변수여도 실제로는 보조 지표로 활용하는 편이 더 적절한 경우가 있습니다. 따라서 변수별 신호 강도와 정책 역할은 반드시 데이터로 검증해야 합니다

이렇게 확정된 변수별 역할과 기준치는 `05_policy_simulation.ipynb`의 정책 매트릭스 설계에 입력값으로 사용됩니다. 다음 단계인 `03_modeling.ipynb`, `04_calibration.ipynb`에서는 PD 설계를 다룹니다.